# Exploratory performance chart

A *nice* exploratory Q×H / Q×P / Q×η chart for one centrifugal pump, built
straight from the `pump` library (no re-implemented physics).

This notebook shares its **dataset** with the bundled PumpFlow project
`pumpflow/examples/explore_performance.pumpflow` and its **Performance
Explorer** node — same light-hydrocarbon FAT (SG ≈ 0.736, 833 m³/h, 73 m), same
ad-hoc guarantee / alternate-duty markers — so the static notebook chart and the
interactive workbench chart tell one story.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pump import Fluid, TestPoint, PerformanceCurve
from pump.utilities.unit_conversion import Q_

# Shared dataset + helpers (Qt-free) — the exact data the .pumpflow file uses.
from pumpflow.sample_data import SINGLE_PUMP_JSON, EXPLORE_POINTS
from pumpflow.persistence import testset_from_json
from pumpflow.binding import row_head_m, row_efficiency_pct, water_density_kgm3

## Build the performance curve from the measured points

Each measured row's `Head = (P_dis − P_suc)/(ρ·g)` and efficiency are computed with
that row's own test-water density and **pinned** onto the `TestPoint` (the
library wants one `Fluid` per curve, so the shared density is just a label).

In [ ]:
tps = testset_from_json(SINGLE_PUMP_JSON)
mean_temp = sum(r.temp_c for r in tps.rows) / len(tps.rows)
test_fluid = Fluid(name="Test water", density=Q_(water_density_kgm3(mean_temp), "kg/m**3"))

points = []
for r in tps.rows:
    head = row_head_m(r, tps.pressure_unit)
    eff = row_efficiency_pct(r, head)
    kwargs = {"_head": Q_(head, "m"), "breaking_power": Q_(r.power_kw, "kW")}
    if eff is not None:
        kwargs["_efficiency"] = Q_(eff, "percent")
    points.append(
        TestPoint(
            fluid=test_fluid,
            capacity=Q_(r.q_m3h, "m**3/h"),
            speed_of_rotation=Q_(r.speed_rpm, "rpm"),
            **kwargs,
        )
    )

curve = PerformanceCurve(test_fluid, points, polynomial_degree=3)

caps = np.asarray(curve.fitter.capacities, dtype=float)
heads = np.asarray(curve.fitter.heads, dtype=float)
powers = np.asarray(curve.fitter.powers, dtype=float)
effs = np.asarray(curve.fitter.efficiencies, dtype=float)
print(f"{len(points)} measured points  ·  Q {caps.min():.0f}–{caps.max():.0f} m³/h")

## Sample smooth curves through the library predictors

`predict_head` / `predict_breaking_power` / `predict_efficiency` evaluate the
fitted degree-3 polynomials — the same fit the API 610 check uses.

In [ ]:
xs = np.linspace(caps.min(), caps.max(), 200)
head_fit = np.array([curve.predict_head(Q_(x, "m**3/h")).to("m").magnitude for x in xs])
power_fit = np.array([curve.predict_breaking_power(Q_(x, "m**3/h")).to("kW").magnitude for x in xs])
eff_fit = np.array([curve.predict_efficiency(Q_(x, "m**3/h")).to("percent").magnitude for x in xs])

# Best efficiency point (BEP) from the fitted efficiency curve.
bep_i = int(np.argmax(eff_fit))
bep_q, bep_eff = xs[bep_i], eff_fit[bep_i]
print(f"BEP ≈ {bep_q:.0f} m³/h  ·  η {bep_eff:.1f}%")

## The exploratory chart

Three stacked, shared-x panels: measured scatter, the degree-3 fit, the BEP, and
the ad-hoc **guarantee / alternate-duty** markers from `EXPLORE_POINTS`.

In [ ]:
HEAD_C, POWER_C, EFF_C, MARK_C = "#2f6fb0", "#b4413c", "#2e7d5b", "#c98a1b"

fig, (ax_h, ax_p, ax_e) = plt.subplots(
    3, 1, sharex=True, figsize=(8, 9),
    gridspec_kw={"height_ratios": [5, 3, 4]},
)
fig.suptitle(f"{tps.pump_tag} — exploratory performance curve", fontsize=12)

for ax, y, yfit, color, label, unit in (
    (ax_h, heads, head_fit, HEAD_C, "Head", "m"),
    (ax_p, powers, power_fit, POWER_C, "Power", "kW"),
    (ax_e, effs, eff_fit, EFF_C, "Efficiency", "%"),
):
    ax.plot(xs, yfit, "-", color=color, lw=2, label="degree-3 fit")
    ax.scatter(caps, y, s=34, color=color, zorder=5, label="measured")
    ax.set_ylabel(f"{label} ({unit})")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc="best")

# BEP guide line across the panels
for ax in (ax_h, ax_p, ax_e):
    ax.axvline(bep_q, color="#5a6573", ls=":", lw=1)
ax_e.plot([bep_q], [bep_eff], "D", color=EFF_C, ms=8, label="BEP")
ax_e.annotate(f"BEP {bep_q:.0f} m³/h", (bep_q, bep_eff), textcoords="offset points",
              xytext=(8, -12), fontsize=8, color="#5a6573")

# ad-hoc markers (shared with the PumpFlow Performance Explorer)
for pt in EXPLORE_POINTS:
    if pt.get("head"):
        ax_h.plot([pt["q"]], [pt["head"]], "*", color=MARK_C, ms=16,
                  markeredgecolor="#7a5510", zorder=6)
        ax_h.annotate(pt["label"], (pt["q"], pt["head"]), textcoords="offset points",
                      xytext=(8, 6), fontsize=8, color="#7a5510")

ax_e.set_xlabel("Capacity  Q (m³/h)")
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()